## Test Setup

### Data
Let's check the /biodata folder needed for the workshop:

In [ ]:
cat("📁 Data Directory Check\n")
cat("========================\n")

biodata_dir <- "/biodata"
expected_paths <- c(
  "resources",
  "resources/Gene_centric_analysis",
  "resources/Gene_centric_analysis/assembly_based_metagenomic_analysis",
  "resources/Genome_assembly",
  "resources/Genome_assembly/checkm",
  "resources/Genome_assembly/fastqc",
  "resources/Genome_assembly/flye_assembly",
  "resources/Genome_assembly/img",
  "resources/Genome_assembly/plassembler",
  "resources/Genome_assembly/prokka",
  "resources/Genome_assembly/raw_reads",
  "resources/MBX_intro",
  "resources/MBX_intro/protect_metadata.csv",
  "resources/MBX_intro/protect_species_abundance.csv",
  "resources/MBX_intro/protect_stool_metabolite_abundance.csv",
  "resources/MGX_assembly",
  "resources/MGX_assembly/assembly",
  "resources/MGX_assembly/raw_reads",
  "resources/MGX_vis",
  "resources/MGX_vis/hmp2_IBD_abd.csv",
  "resources/MGX_vis/hmp2_IBD_metadata.csv",
  "resources/MetaGEAR_vis",
  "resources/MetaGEAR_vis/nar_operon_query_output",
  "resources/MetaPhlan_intro",
  "resources/MetaPhlan_intro/Metaphlan4_DB",
  "resources/MetaPhlan_intro/mgx_reads",
  "resources/MetaPhlan_intro/reference_based_metagenomic_analysis",
  "resources/Multivariate_analysis",
  "resources/Multivariate_analysis/demo_output1",
  "resources/Multivariate_analysis/demo_output2",
  "resources/Multivariate_analysis/task1",
  "resources/Multivariate_analysis/task2",
  "resources/Protein_structure_analysis",
  "resources/Protein_structure_analysis/input"
)

missing_paths <- expected_paths[!file.exists(file.path(biodata_dir, expected_paths))]

if (!dir.exists(biodata_dir)) {
  cat("❌", biodata_dir, "is not mounted -- check the devcontainer 'mounts' config\n")
} else if (length(missing_paths) == 0) {
  cat("✅ All", length(expected_paths), "expected resources found under", biodata_dir, "\n")
} else {
  cat("❌ Missing", length(missing_paths), "of", length(expected_paths), "expected paths under", biodata_dir, ":\n")
  for (p in missing_paths) cat("  -", p, "\n")
}

### R Environment
Check that R and required packages are properly configured

In [ ]:
# Check R version and basic information
cat("🔍 R Environment Information\n")
cat("==========================\n")
cat("R version:", R.version.string, "\n")
cat("R home directory:", R.home(), "\n")
cat("Platform:", R.version$platform, "\n")
cat("✅ R is working correctly!\n")

Check R library paths and available packages

In [ ]:
cat("\n📚 R Library Information\n")
cat("========================\n")
cat("Library paths:\n")
for (path in .libPaths()) {
  cat("  -", path, "\n")
}

cat("\nTotal installed packages:", length(installed.packages()[,1]), "\n")

Check system capabilities and configuration

In [ ]:
cat("\n⚙️  System Configuration\n")
cat("=======================\n")
cat("Working directory:", getwd(), "\n")
#cat("Available memory:", round(memory.size(max = TRUE), 2), "MB\n")
cat("Number of CPU cores:", parallel::detectCores(), "\n")

# Check if we can create files in the working directory
test_file <- tempfile()
cat("Write permissions test:", ifelse(file.create(test_file), "✅ OK", "❌ Failed"), "\n")
if (file.exists(test_file)) file.remove(test_file)

Check if required packages are available in the dev container. All packages should be pre-installed (fossil is not available on Mac)

In [ ]:
cat("\n📦 Package Availability Check\n")
cat("=============================\n")

required_packages <- c("ggplot2", "vegan", "stringr", "ggpubr",
                      "tidyverse", "fossil", "BiocManager", "Maaslin2")

for (pkg in required_packages) {
  if (require(pkg, character.only = TRUE, quietly = TRUE)) {
    cat("✅", pkg, "is available\n")
  } else {
    cat("❌", pkg, "is missing from the environment\n")
  }
}

## Basic data manipulation and vizualization

In [ ]:
# Create sample data
sample_data <- data.frame(
  species = c("Escherichia coli", "Bacillus subtilis", "Staphylococcus aureus", "Pseudomonas aeruginosa"),
  count = c(150, 89, 234, 67),
  temperature = c(37, 30, 37, 28)
)

print("Sample microbiology data:")
print(sample_data)

# String manipulation with stringr
library(stringr)
sample_data$genus <- str_extract(sample_data$species, "^[A-Z][a-z]+")
sample_data$species_short <- str_c(str_sub(sample_data$genus, 1, 1), ". ",
                                   str_extract(sample_data$species, " [a-z]+$"))

print("\nData with extracted genus and short species names:")
print(sample_data[, c("species", "genus", "species_short", "count")])

In [ ]:
library(ggplot2)

# Create a bar plot of bacterial counts
p1 <- ggplot(sample_data, aes(x = species_short, y = count, fill = genus)) +
  geom_col() +
  labs(title = "Bacterial Colony Counts",
       x = "Species",
       y = "Colony Count",
       fill = "Genus") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

# Create a scatter plot of count vs temperature
p2 <- ggplot(sample_data, aes(x = temperature, y = count, color = genus)) +
  geom_point(size = 4) +
  geom_smooth(method = "lm", se = FALSE, color = "gray50") +
  labs(title = "Colony Count vs Growth Temperature",
       x = "Temperature (°C)",
       y = "Colony Count",
       color = "Genus") +
  theme_minimal()

# Display both plots side by side with enough room for titles and legends
options(repr.plot.width = 16, repr.plot.height = 8)
grid::grid.newpage()
grid::pushViewport(grid::viewport(layout = grid::grid.layout(1, 2)))
print(p1, vp = grid::viewport(layout.pos.row = 1, layout.pos.col = 1))
print(p2, vp = grid::viewport(layout.pos.row = 1, layout.pos.col = 2))
grid::popViewport()